In [ ]:
pip install catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q3_path = os.path.join(path, 'Q3_data.csv')
Q3_data = pd.read_csv(Q3_path)

In [ ]:
# Task 2: Write your code here:
Q3_data.head()

In [ ]:
# Task 3: Write your code here:
Q3_data.info()

In [ ]:
# Task 4: Write your code here:
Q3_data.describe()

In [ ]:
# Task 1: Write your code here:
categorical_cols = Q3_data.select_dtypes(include=["object"]).columns
num_cols = Q3_data.select_dtypes(include=["int64", "float64"]).columns

for col in categorical_cols:
    Q3_data[col] = Q3_data[col].fillna(Q3_data[col].mode()[0])
for col in num_cols:
    Q3_data[col] = Q3_data[col].fillna(Q3_data[col].mean())


Q3_data.head()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(Q3_data)
Q3_data.info()

In [ ]:
#there is no categorical columns but i wrote the code just in case
print(categorical_cols)

In [ ]:
# Task 3: Write your code here:
#in comment because we dont need it
#for col in categorical_cols:

  #le = LabelEncoder()
  #Q3_data[col] = le.fit_transform(Q3_data[col].astype(str))



In [ ]:
# Task 4: Write your code here:
# no scaling for my target
features_cols = Q3_data.select_dtypes(include=["int64", "float64", "object"]).columns.drop("Target")
scaler = StandardScaler()
Q3_data[features_cols] = scaler.fit_transform(Q3_data[features_cols])

Q3_data.head()

In [ ]:
# Task 5: Write your code here:
Q3_data["Target"].hist()
# it is clearly inbalanced

In [ ]:
# Task 1: Write your code here:
X = Q3_data[features_cols]
y = Q3_data["Target"]

X.info()


In [ ]:
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_accuracy = []
lr_f1 = []

model =  CatBoostClassifier(
      verbose=0,
      n_estimators=20,
      max_depth=4)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)


    lr_accuracy.append(accuracy)
    lr_f1.append(f1)
    # showing class distribution






print("LOGISTIC REGRESSION Performance:")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1:  {np.mean(lr_f1):.4f}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': features_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# to get the feature at the top since the graghp dosnt show it
print(feature_importance["feature"][0])

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(feature_importance["feature"][0])


In [ ]:
# Task Bonus: Write your code here:
X_new = X["P_2"]
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_accuracy = []
lr_f1 = []

model =  CatBoostClassifier(
      verbose=0,
      n_estimators=20,
      max_depth=4)

                                                    # use the singel feature only but same y
for fold, (train_idx, test_idx) in enumerate(skf.split(X_new, y), start=1):
    # indexing for each fold
    X_train_new, X_test_new = X.iloc[train_idx], X.iloc[test_idx]
    y_train_new, y_test_new = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train_new, y_train_new)
    y_pred_new = model.predict(X_test_new)

    accuracy = accuracy_score(y_test_new, y_pred_new)
    f1 = f1_score(y_test_new, y_pred_new, zero_division=0)


    lr_accuracy.append(accuracy)
    lr_f1.append(f1)



print("LOGISTIC REGRESSION Performance:")
print(f"  Accuracy:  {np.mean(lr_accuracy):.4f}")
print(f"  F1:  {np.mean(lr_f1):.4f}")



In [ ]:
# with all features Accuracy:  0.8363
#  F1:  0.6827
# as you can see we got same acc and F1 score even with using 1 featuer only